# PersonaNeedle — Eğitim Zinciri (B → C → D → E)

Bu notebook PersonaNeedle modelini **Google Colab T4 GPU** üzerinde sıfırdan eğitir,
quantize eder, 495 personayı paketler ve M1–M60 serisini doğrular.

**Önce:** Runtime → Change runtime type → **T4 GPU** seçili olmalı.

**Dürüstlük notu:** Gerçek API key'leri girilmezse veri üretimi `persona_math` fallback'ine
düşer ve tier **`Ölçülmüş` OLMAZ** (`Simülasyon` kalır). `Ölçülmüş` tier'ı yalnızca gerçek
(eğitilmiş) model ölçümü açar — bu projenin değişmez kuralı.

| Hücre | Görev | Süre (T4) |
|---|---|---|
| 1–2 | Kurulum + API key | ~3 dk |
| 3–4 | **B** — veri üretimi (~9,900 örnek) | 2–4 saat |
| 5–6 | **C** — fine-tune + INT4/GGUF | 4–6 saat |
| 7 | **D** — 495 persona bundle | ~15 dk |
| 8 | **E** — validasyon + M61 | ~10 dk |
| 9 | Push | ~1 dk |


## Hücre 1 — Kurulum
Repoyu klonlar ve bağımlılıkları kurar (teacher SDK'ları + jsonlines dahil).

**Beklenen:** `Successfully installed ...`, hata yok.
**Hata olursa:** `pip` çakışması → çalışma zamanını yeniden başlat (Runtime → Restart) ve tekrar çalıştır.

In [ ]:
!git clone https://github.com/mk350174-cmd/Persona.git
%cd Persona
!pip install -q -r needle/training/requirements.txt
!pip install -q -r needle/finetune/requirements.txt
!pip install -q groq google-generativeai openai jsonlines

## Hücre 2 — API Key'leri
$0 hibrit strateji: **AIML (0.5) + Groq (0.3) + Gemini (0.2)**. En az biri yeterli.
Key alma: aimlapi.com · console.groq.com · aistudio.google.com

**Beklenen:** Sessiz (assert geçer).
**Hata olursa:** Hiç key yoksa assert düşer — en az bir key gir (yoksa `persona_math` fallback, `Ölçülmüş` olmaz).

In [ ]:
import os
os.environ["AIMLAPI_KEY"]  = ""   # buraya yaz
os.environ["GROQ_API_KEY"] = ""   # buraya yaz
os.environ["GEMINI_API_KEY"] = "" # buraya yaz

assert any(os.environ.get(k) for k in ("AIMLAPI_KEY", "GROQ_API_KEY", "GEMINI_API_KEY")), \
    "En az bir API key gir — yoksa persona_math fallback'e duser ve tier Olculmus OLMAZ."
print("Aktif key'ler:", [k for k in ("AIMLAPI_KEY","GROQ_API_KEY","GEMINI_API_KEY") if os.environ.get(k)])

## Hücre 3 — GÖREV B: Eğitim Verisi Üretimi
Hibrit öğretmenlerle 495 persona × 20 konuşma = **~9,900 örnek** üretir. Her 50 personada
checkpoint yazar; kesilirse `--resume` ile kaldığı yerden devam eder.

**Beklenen:** `active teachers: ['aiml', 'groq', 'gemini']` + `ceid_samples` sayısı artar; sonunda
`train/val/test.jsonl` + `drift/voice_dataset.jsonl`. **Süre:** 2–4 saat (Groq ana kaynak).
**Hata olursa:** Rate limit → otomatik diğer teacher'a/persona_math'e düşer; oturum kesilirse bu hücreyi `--resume` ile tekrar çalıştır (tamamlananları atlar).

In [ ]:
!python -m needle.training.pipeline \
  --aiml-key $AIMLAPI_KEY \
  --groq-key $GROQ_API_KEY \
  --gemini-key $GEMINI_API_KEY \
  --n-conversations 20 --resume

## Hücre 4 — Veri Kontrolü
Üretilen eğitim setinin yeterli olduğunu doğrular.

**Beklenen:** `Train: ~7900 örnek` (0.8 split) ve örnek anahtarları
(`persona_id, conversation, k_layer_vector, ceid_labels, confidence`).
**Hata olursa:** `assert` düşerse veri üretimi yarım — Hücre 3'ü `--resume` ile tekrar çalıştır.

In [ ]:
import jsonlines
with jsonlines.open("needle/training/data/train.jsonl") as f:
    train = list(f)
print(f"Train: {len(train)} ornek")
assert len(train) > 5000, "Yetersiz veri — Hucre 3'u --resume ile tekrar calistir"
print("Ornek anahtarlari:", list(train[0].keys()))

## Hücre 5 — GÖREV C: Fine-tune (LoRA, T4 GPU)
Birleşik kayıp `0.4·CEID + 0.3·drift + 0.3·voice`, LoRA (~0.8M eğitilebilir), 3 epoch.
En iyi model `needle/finetune/checkpoints/best_model/` altına kaydedilir ve `untrained=False` olur.

**Beklenen (epoch 3 hedefleri):** CEID MAE < 0.05 · Drift F1 > 0.88 · Perplexity < 25.
**Hedef tutmazsa:** MAE>0.05 → `--epochs 5`; F1<0.88 → (DriftLoss `pos_weight` 3.0); PPL>25 → (VoiceLoss `label_smoothing` 0.05).
**Bellek hatası:** `--batch-size 8`. **Oturum kesilirse:** checkpoint'ten devam.

In [ ]:
!python -m needle.finetune.trainer \
  --train needle/training/data/train.jsonl \
  --val needle/training/data/val.jsonl \
  --lora --epochs 3 --device cuda \
  --batch-size 16 --gradient-accumulation 4 --warmup-steps 100 \
  --checkpoint-dir needle/finetune/checkpoints/

## Hücre 6 — INT4 Quantization + GGUF Export
FP32 (~105MB) → INT4 (~13MB). `gguf_writer` config'i otomatik `--quantized` klasöründen okur
(int4 oraya `config.json` yazar).

**Beklenen:** `compression_ratio ≈ 0.125` ve `model.gguf ≈ 13MB`.
**GGUF > 20MB ise:** `int4 --group-size 64` ile yeniden quantize et.

In [ ]:
!python -m needle.finetune.quantize.int4 \
  --model needle/finetune/checkpoints/best_model/ \
  --output needle/finetune/quantize/output/ \
  --group-size 128

!python -m needle.finetune.quantize.gguf_writer \
  --quantized needle/finetune/quantize/output/ \
  --output needle/finetune/quantize/output/model.gguf

!ls -lh needle/finetune/quantize/output/model.gguf

## Hücre 7 — GÖREV D: 495 Persona Bundle
Tek eğitilmiş GGUF tüm personalar tarafından paylaşılır (PersonaNeedle persona'yı K-layer
prefix ile koşullar). `--gguf` verildiği için bundle'lar **`untrained: false`** olur.

**Beklenen:** `bundled: 495`, `untrained: 0`, `catalog.json` güncellenir.
**Hata olursa:** `--workers 2`'ye düşür; kesilirse `--resume` ile devam (var olanları atlar).

In [ ]:
!python -m needle.pipeline.bulk_bundler \
  --gguf needle/finetune/quantize/output/model.gguf \
  --workers 4 --resume

## Hücre 8 — GÖREV E: Akademik Validasyon + M61
Eğitilmiş modelle M1–M60 yeniden ölçülür ve tier `Simülasyon → Ölçülmüş` yükseltilir;
M61 sonuçları checkpoint'le güncellenir; rapor üretilir.

**Beklenen (REPORT.md):** `Measured: 51+` (eğitim gerçekten yapıldıysa).
**Not:** Model eğitilmemişse `validator` `persona_math` fallback kullanır ve tier `Ölçülmüş` OLMAZ — bu beklenen dürüst davranıştır.

In [ ]:
!python -m validation.validator --all --workers 2
!python papers/M61_experiments/exp_m61.py \
  --checkpoint needle/finetune/checkpoints/best_model/
!python -m validation.report_generator --output validation/REPORT.md
!cat validation/REPORT.md

## Hücre 9 — Push
Sadece hafif/anlamlı artefaktları push'lar (catalog.json, REPORT.md, results/M61/).
JSONL veri, checkpoint ve bundle alt-klasörleri `.gitignore`'da — push'lanmaz.

**Önce:** e-posta/isim ve GitHub kimlik doğrulaması (token) ayarlı olmalı.
**Beklenen:** `claude/inspiring-hopper-1qnKP` dalına başarılı push. Sonra PR aç → main.

In [ ]:
import subprocess
subprocess.run(["git", "config", "user.email", "your@email.com"])
subprocess.run(["git", "config", "user.name", "Your Name"])
subprocess.run(["git", "add",
                "needle/bundles/catalog.json",
                "validation/REPORT.md",
                "results/M61/"])
subprocess.run(["git", "commit", "-m",
                "feat: PersonaNeedle trained, 495 bundled, M1-M60 Measured"])
subprocess.run(["git", "push", "origin", "claude/inspiring-hopper-1qnKP"])